In [5]:
import os
import glob
import pandas as pd
import re
import shutil
import numpy as np

In [6]:
# --- TUNING ---

USE_DUMMY_DATA = False

In [7]:
#--- HELPER FUNCTIONS ---

# Calculates roll stiffness, distribution and damping parameters
def calculate_mechanical_metrics(lap_setup_data):
    # Read values directly from the lap setup data series
    k_sf = lap_setup_data['k_sf']
    k_sr = lap_setup_data['k_sr']
    k_ARbf = lap_setup_data['k_ARBf']
    k_ARbr = lap_setup_data['k_ARBr']
    d_fb = lap_setup_data['dclicks_fb']
    d_fr = lap_setup_data['dclicks_fr']
    d_rb = lap_setup_data['dclicks_rb']
    d_rr = lap_setup_data['dclicks_rr']

    # Apply the new formulas
    roll_stiffness = k_sf + k_sr + k_ARbf + k_ARbr
    roll_stiffness_dist = k_sf + k_ARbf - k_sr - k_ARbr
    pitch_damping = d_fb + d_rr - d_fr - d_rb
    heave_damping = d_fb + d_rb - d_fr - d_rr

    return {
        'roll_stiffness': roll_stiffness,
        'roll_stiffness_dist': roll_stiffness_dist,
        'pitch_damping': pitch_damping,
        'heave_damping': heave_damping
    }


In [8]:
# --- MAIN SCRIPT ---

def find_target_track_name(base_path, data_type):
    search_path = os.path.join(base_path, data_type)
    target_folders = glob.glob(os.path.join(search_path, "Target_*"))

    if not target_folders:
        raise FileNotFoundError(f"No 'Target_*' directory found in '{search_path}'.")
    
    if len(target_folders) > 1:
        raise ValueError(f"Multiple 'Target_*' directories found. Please specify which one to use. Found: {target_folders}")
    
    target_folder_name = os.path.basename(target_folders[0])
    return target_folder_name.replace("Target_", "")

# Main function
def process_telemetry():
    print("Starting telemetry processing...")

    data_type = "Dummy" if USE_DUMMY_DATA else "Real"
    telemetry_base_path = os.path.join("Telemetry", data_type)
    sections_base_path = os.path.join("Sections", data_type)

    print(f"Data type set to: '{data_type}'")

    try:
        target_track_name_caps = find_target_track_name("Sections", data_type)
    except (FileNotFoundError, ValueError) as e:
        print(f"Error: {e}")
        return
    
    try:
        target_sections_path = os.path.join(sections_base_path, f"sections_target_{target_track_name_caps.lower()}.csv")
        target_sections_df = pd.read_csv(target_sections_path)
        print(f"Successfully loaded target sections from '{target_sections_path}'.")
    except FileNotFoundError:
        print(f"FATAL: Target section definitions not found at '{target_sections_path}'. This file is now required. Aborting.")
        return

    target_section_defs_path = os.path.join(sections_base_path, f"Target_{target_track_name_caps}")
    output_base_path = os.path.join(telemetry_base_path, f"Target_{target_track_name_caps}")

    if os.path.exists(output_base_path):
        print(f"\n! WARNING: Output directory '{output_base_path}' already exists.")
        choice = input("   Do you want to REMOVE all existing files in this directory and continue? (y/n): ").lower()
        if choice in ['y', 'yes']:
            shutil.rmtree(output_base_path)
            print("   > Directory deleted successfully.")
        else:
            print("   Operation cancelled by user. Aborting script.")
            return
        
    print(f"Detected Target track: '{target_track_name_caps}'")
    print(f"Reading section definitions from: '{target_section_defs_path}'")
    print(f"Output will be saved to '{output_base_path}'")

    delta_report_data = []

    target_section_files = sorted(glob.glob(os.path.join(target_section_defs_path, "section*.csv")))

    if not target_section_files:
        print(f"WARNING: No section definition files ('section*.csv') found in '{target_section_defs_path}'.")
        return
    
    for section_def_file in target_section_files:
        target_section_num = int(re.search(r'section(\d+)', os.path.basename(section_def_file)).group(1))
        print(f"\n--- Processing Target Section {target_section_num:02d} ---")

        try:
            target_section_info = target_sections_df[target_sections_df['segment_id'] == target_section_num].iloc[0]
            target_distance = target_section_info['end_m'] - target_section_info['start_m']
        except (IndexError, KeyError):
            print(f"   ! Warning: Could not find definition for target segment_id {target_section_num}. Skipping section.")
            continue # Corrected to 'continue'
        
        matches_df = pd.read_csv(section_def_file)
       
        for index, row in matches_df.iterrows():
            source_track, source_segment_id, certainty = row['track_name'], row['segment_id'], row['certainty']
            print(f"   > Match found: Track '{source_track}' , Segment ID '{source_segment_id}'")

            try:
                setup_path = os.path.join("Setup", data_type, f"{source_track}.csv")
                setup_df = pd.read_csv(setup_path)
                setup_df.columns = setup_df.columns.str.strip()
                
                ### MODIFIED ###
                # This line is CRITICAL for looking up lap data. It was missing from your script.
                setup_df = setup_df.set_index('lap')
                
                source_sections_file = os.path.join(sections_base_path, f"sections_{source_track}.csv")                                                            
                all_source_segments_df = pd.read_csv(source_sections_file)
                segment_info = all_source_segments_df[all_source_segments_df['segment_id'] == source_segment_id]
                if segment_info.empty:
                    print(f"   ! Error: Segment ID {source_segment_id} not found in '{source_sections_file}'. Skipping.")
                    continue
                start_m, end_m = segment_info.iloc[0]['start_m'], segment_info.iloc[0]['end_m']

                source_lap_files = sorted(glob.glob(os.path.join(telemetry_base_path, source_track, f"lap*_tele_*.csv")))
                if not source_lap_files:
                    print(f"   ! Warning: No lap files found for track '{source_track}' in '{os.path.join(telemetry_base_path, source_track)}'.")
                    continue
                
                ### ADDED ###
                # This logic is required to correctly calculate distance and slice data for wrap-around sections.
                temp_lap_df = pd.read_csv(source_lap_files[0])
                total_lap_distance = temp_lap_df['Distance'].iloc[-1]
                
                if start_m > end_m: # Wrap-around case
                    training_distance_m = (total_lap_distance - start_m) + end_m
                    print(f"     * Detected wrap-around section. Training distance: {training_distance_m:.2f}m")
                else: # Normal case
                    training_distance_m = end_m - start_m
               
            except FileNotFoundError as e:
                print(f"   ! Error: Source file not found: {e}. Skipping")
                continue
            except Exception as e:
                print(f"   ! Error during setup for match '{source_track}': {e}. Skipping match.")
                continue

            output_dir = os.path.join(output_base_path, f"Section{target_section_num:02d}", f"{source_track}_segment{source_segment_id}")
            os.makedirs(output_dir, exist_ok=True)

            for lap_file in source_lap_files:
                ### MODIFIED ###
                # This is the PRIMARY FIX. The regex now correctly matches your filenames (e.g., 'lap1_tele_...').
                lap_num_match = re.search(r'lap(\d+)_tele_', os.path.basename(lap_file))
                if not lap_num_match:
                    continue # If the filename doesn't match, skip it.
                lap_num_str = lap_num_match.group(1)

                try:
                    try:
                        lap_setup_data = setup_df.loc[int(lap_num_str)]
                    except KeyError:
                        raise ValueError(f"FATAL: No setup data found for lap {lap_num_str} in '{setup_path}'. Please correct the setup file.")
                        
                    full_lap_df = pd.read_csv(lap_file)
                    
                    ### MODIFIED ###
                    # This logic correctly slices the telemetry data for both normal and wrap-around sections.
                    if start_m > end_m: # Wrap-around case
                        part1_df = full_lap_df[full_lap_df['Distance'] >= start_m]
                        part2_df = full_lap_df[full_lap_df['Distance'] < end_m]
                        segment_df = pd.concat([part1_df, part2_df])
                    else: # Normal case
                        segment_df = full_lap_df[(full_lap_df['Distance'] >= start_m) & (full_lap_df['Distance'] < end_m)]

                    if not segment_df.empty:
                        output_filename = os.path.join(output_dir, f"lap_{lap_num_str}.csv")
                        segment_df.to_csv(output_filename, index=False)
                        print(f"     + Created: '{output_filename}'")

                        total_lap_duration = full_lap_df['Time'].iloc[-1]

                        ### MODIFIED ###
                        # This logic correctly calculates the section duration for both cases.
                        if start_m > end_m:
                            if not part1_df.empty and not part2_df.empty:
                                section_duration = (total_lap_duration - part1_df['Time'].iloc[0]) + part2_df['Time'].iloc[-1]
                            else: section_duration = 0
                        else:
                            section_duration = segment_df['Time'].iloc[-1] - segment_df['Time'].iloc[0]

                        ### MODIFIED ###
                        # Added pd.to_numeric to handle the '+' sign and convert the delta string to a number.
                        best_time_delta_from_setup = pd.to_numeric(lap_setup_data['best_time_delta'])
                        effective_time_delta = best_time_delta_from_setup * (section_duration / total_lap_duration) if total_lap_duration > 0 else 0

                        ### MODIFIED ###
                        # Switched from per_km to per_meter for a more standard unit.
                        time_delta_per_km = effective_time_delta / training_distance_m * 10**3 if training_distance_m > 0 else 0

                        mech_metrics = calculate_mechanical_metrics(lap_setup_data)
                        aero_metrics = {
                            'downforce': lap_setup_data['downforce'],
                            # Converts CoP from percentage (e.g. 35.9) to ratio (0.359)
                            'cop': lap_setup_data['cop']
                        }

                        report_row = {
                            'target_track_section': target_section_num,
                            'certainty': certainty,
                            'training_track': source_track,
                            'training_track_section': int(source_segment_id),
                            'lap': int(lap_num_str),
                            'target_distance': int(round(target_distance)),
                            'training_distance': int(round(training_distance_m)),
                            'section_time': section_duration,
                            'time_delta': effective_time_delta,
                            'time_delta_per_km': time_delta_per_km,
                            **mech_metrics,
                            **aero_metrics
                        }
                        delta_report_data.append(report_row)
                
                except (ValueError, KeyError) as e:
                    print(f"   ! {e}")
                    print("   Aborting script due to critical data error.")
                    return                          
                except Exception as e:
                    print(f"   ! Error processing lap file '{lap_file}': {e}")
          
    if delta_report_data:
        print("\n--- Generating Final Report ---")
        try:
            report_df = pd.DataFrame(delta_report_data)
            report_df.sort_values(
                by=['target_track_section', 'certainty', 'training_track', 'training_track_section', 'lap'],
                ascending=[True, False, True, True, True],
                inplace=True
            )

            report_df['target_track_section'] = report_df['target_track_section'].apply(lambda x: f"{x:02d}")
            for col in ['certainty', 'roll_stiffness_dist']:
                if col in report_df.columns: report_df[col] = report_df[col].map('{:.3f}'.format)

            ### MODIFIED ###
            # Corrected formatting to use the new 'time_delta_per_km' column.
            if 'time_delta' in report_df.columns:
                report_df['time_delta'] = report_df['time_delta'].map('{:+.3f}'.format)
            if 'section_time' in report_df.columns:
                report_df['section_time'] = report_df['section_time'].map('{:.3f}'.format)
            if 'time_delta_per_km' in report_df.columns:
                report_df['time_delta_per_km'] = report_df['time_delta_per_km'].map('{:+.6f}'.format)

            ### MODIFIED ###
            # Corrected the list of columns to format, removing old metric names and adding new ones.
            for col in ['roll_stiffness', 'pitch_damping', 'heave_damping', 'downforce', 'cop']:
                if col in report_df.columns: report_df[col] = report_df[col].map('{:.1f}'.format)
            
            # Removed distance columns from formatting as they are integers.

            ### MODIFIED ###
            # Corrected the list of final columns to match the new metrics being calculated.
            final_columns = [
                'target_track_section', 'certainty', 'training_track', 'training_track_section', 'lap',
                'target_distance', 'training_distance', 'section_time',
                'roll_stiffness', 'roll_stiffness_dist', 'pitch_damping', 'heave_damping',
                'downforce', 'cop',
                'time_delta', 'time_delta_per_km',
            ]

            report_df = report_df[[col for col in final_columns if col in report_df.columns]]

            ### MODIFIED ###
            # Changed filename to be more descriptive.
            report_filename = os.path.join(output_base_path, "final_analysis_report.csv")
            report_df.to_csv(report_filename, index=False)
            print(f"   + Success! Report saved to '{report_filename}'")
        except Exception as e:
            print(f"   ! Error creating the final report: {e}")

    print("\n--- Telemetry processing complete. ---")

if __name__ == '__main__':
    # Make sure to include the helper function in your final script file
    process_telemetry()

Starting telemetry processing...
Data type set to: 'Real'
Successfully loaded target sections from 'Sections\Real\sections_target_monza.csv'.

! WARNING: Output directory 'Telemetry\Real\Target_Monza' already exists.


OSError: [WinError 145] The directory is not empty: 'Telemetry\\Real\\Target_Monza\\Section05'